In [1]:
import pandas as pd
import numpy as np
import torch
from datasets import Dataset, DatasetDict, ClassLabel, Features
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    pipeline
)
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

2025-11-16 17:20:34.531543: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1763313634.984040      48 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1763313635.112816      48 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

In [ ]:
import transformers
transformers.__version__

In [25]:
df = pd.read_csv("/kaggle/input/legal-indian-contract-clauses-dataset/legal_contract_clauses.csv")

In [ ]:
TEXT_COLUMN = "clause_text"
LABEL_COLUMN = "risk_level"

In [ ]:
print("Step 2: Creating label mappings...")

# Get all unique labels
labels = df[LABEL_COLUMN].unique()
labels.sort() # Sort for consistency

# Create mappings between text labels and integer IDs
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for i, label in enumerate(labels)}
num_labels = len(labels)

# Add integer 'label' column to the DataFrame
df['label'] = df[LABEL_COLUMN].map(label2id)

In [ ]:
# --- 4. Convert to Hugging Face Dataset ---
print("Step 3: Converting to Hugging Face Dataset...")
dataset = Dataset.from_pandas(df)

# --- NEW: Cast 'label' column to ClassLabel ---
# This is required for stratification to work.
# It tells the 'datasets' library that 'label' is a category, not just an integer.
print("Step 3a: Casting 'label' column to ClassLabel for stratification...")

# Recreate the list of label names in the correct integer order
# (Uses the 'id2label' dictionary we created in Step 3)
label_names_in_order = [id2label[i] for i in range(num_labels)]

# Create the ClassLabel feature
class_label_feature = ClassLabel(names=label_names_in_order)

# Cast the 'label' column in the dataset
dataset = dataset.cast_column("label", class_label_feature)

print(f"Dataset features after casting: {dataset.features}")
# --- END NEW ---


# Split the dataset (e.g., 80% train, 20% test)
# This line will now work correctly
print("Step 3b: Splitting dataset...")
train_test_split = dataset.train_test_split(test_size=0.2, seed=42, stratify_by_column="label")

dataset_dict = DatasetDict({
    'train': train_test_split['train'],
    'test': train_test_split['test']
})

print(f"Training data: {len(dataset_dict['train'])} examples")
print(f"Test data: {len(dataset_dict['test'])} examples")

In [ ]:
print("\nStep 4: Loading model and tokenizer (nlpaueb/legal-bert-base-uncased)...")
model_name = "nlpaueb/legal-bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id
)

In [ ]:
print("\nStep 5: Tokenizing dataset...")

def tokenize_function(examples):
    # This tokenizes the text, padding and truncating
    return tokenizer(examples[TEXT_COLUMN], padding="max_length", truncation=True)

# Apply tokenization to the entire dataset (this will be fast)
tokenized_datasets = dataset_dict.map(tokenize_function, batched=True)

# Remove the original text columns, as the model only needs token IDs
columns_to_remove = [TEXT_COLUMN, LABEL_COLUMN]
if "__index_level_0__" in tokenized_datasets["train"].column_names:
    columns_to_remove.append("__index_level_0__")
    
tokenized_datasets = tokenized_datasets.remove_columns(columns_to_remove)
tokenized_datasets.set_format("torch")

print("Tokenization complete.")

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    precision = precision_score(labels, predictions, average='weighted')
    recall = recall_score(labels, predictions, average='weighted')
    
    return {
        "accuracy": acc,
        "f1": f1,
        "precision": precision,
        "recall": recall
    }

In [ ]:
# --- 8. Set Training Arguments & Train ---
print("\nStep 6: Setting training arguments...")

# Define the output directory for the fine-tuned model
model_output_dir = "./legal_bert_finetuned_risk"

training_args = TrainingArguments(
    output_dir=model_output_dir,
    learning_rate=2e-5,
    per_device_train_batch_size=8,  # Adjust down (4, 8) if you get CUDA OOM errors
    per_device_eval_batch_size=8,
    num_train_epochs=3, # 3 epochs is a good starting point
    weight_decay=0.01,
    load_best_model_at_end=True,
    push_to_hub=False,
    report_to="none", # Disables wandb logging
    eval_strategy="epoch",
    save_strategy="epoch",
)

# Initialize the Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [ ]:
print("\nStep 7: Starting model training (this may take a while)...")
# Start training!
trainer.train()

print("Training finished.")

# Save the final model and tokenizer
trainer.save_model(model_output_dir)
tokenizer.save_pretrained(model_output_dir)
print(f"Model saved to {model_output_dir}")

In [ ]:
print("\nStep 8: Using the fine-tuned model for inference...")

# Load the fine-tuned model from the directory where we saved it
finetuned_model_path = "./legal_bert_finetuned_risk"
risk_classifier = pipeline(
    "text-classification", 
    model=finetuned_model_path,
    # Use id 0 if no GPU, or 0, 1, etc. for specific GPUs
    device=0 if torch.cuda.is_available() else -1 
)

print("Fine-tuned model loaded into a pipeline.")

In [ ]:
test_clause = "Indemnification. The Contractor (the 'Indemnitor') agrees to indemnify, defend, and hold harmless the Client... regardless of whether such claim is caused in whole or in part by the negligence... of the Client."

result = risk_classifier(test_clause, return_all_scores=True)
print(f"\nTest Clause: '{test_clause[:100]}...'\n")
print(f"Prediction: {result}")

# Test with a low-risk clause
test_clause_low = "This settlement agreement is reached amicably between both parties."
result_low = risk_classifier(test_clause_low)
print(f"\nTest Clause: '{test_clause_low}'\n")
print(f"Prediction: {result_low}")

In [32]:
test_clause = df.iloc[5].clause_text

In [33]:
test_clause

'The  Distributor  shall  not order or                   purchase Products from any source other than the Company.'

In [2]:
print("Loading summarization model...")
try:
    summarizer_pipeline = pipeline(
        "summarization",
        model="t5-small"
    )
    print("✅ T5-small summarizer model loaded")
except Exception as e:
    print(f"Failed to load T5-small, summarization will be disabled. Error: {e}")
    summarizer_pipeline = None

Loading summarization model...


config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

Device set to use cpu


✅ T5-small summarizer model loaded


In [22]:
def generate_clause_summary(text):
        """Generate a short summary of the clause"""
        if not summarizer_pipeline:
            return "Summarization model not available."
        
        if not text:
            return "No text to summarize."
        
        try:
            # T5 models expect a prefix for the task
            input_text = "summarize: " + text
            
            summary_result = summarizer_pipeline(
                input_text,
                max_new_tokens=80,
                min_length=15,  # Min 15 tokens
                do_sample=False
            )
            return summary_result[0]['summary_text']
        except Exception as e:
            print(f"Error in summarization: {e}")
            return "Summarization failed."

In [34]:
print(generate_clause_summary(test_clause))

Your max_length is set to 200, but your input_length is only 23. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=11)


the Distributor shall not order or purchase Products from any source other than the Company.
